In [1]:
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error
from xgboost import XGBClassifier, XGBRegressor
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical


In [2]:
breast = pd.read_csv('breast-level_annotations.csv')
columns = ['study_id','series_id', 'laterality', 'view_position',
       'breast_birads', 'breast_density']
breast = breast[columns]
breast['breast_density'] = breast['breast_density'].map({'DENSITY A':1 , 'DENSITY B': 2 , 'DENSITY C' : 3 , 'DENSITY D': 4})

# Convert BI-RADS to numeric (int)
breast['breast_birads'] = (
    breast['breast_birads']
    .str.replace('BI-RADS ', '', regex=False)  # Remove prefix
    .astype(int)  # Convert to integer
)

# Collapse to breast-level (one row per study_id + laterality)
breast = breast.groupby(['study_id','series_id', 'laterality'], as_index=False).agg({
    'breast_birads': 'max',
    'breast_density': 'max'
})

breast.head()

,study_id,series_id,laterality,breast_birads,breast_density
0,0025a5dc99fd5c742026f0b2b030d3e9,47d59b788d64eecab165d97471c4131a,L,1,3
1,0025a5dc99fd5c742026f0b2b030d3e9,47d59b788d64eecab165d97471c4131a,R,1,3
2,0028fb2c7f0b3a5cb9a80cb0e1cdbb91,ff5d6ba6e303628092020e897fcbc9b1,L,2,3
3,0028fb2c7f0b3a5cb9a80cb0e1cdbb91,ff5d6ba6e303628092020e897fcbc9b1,R,2,3
4,0034765af074f93ed33d5e8399355caf,91de7075b7eb54cc4fc959ef67b9df56,L,2,3


In [3]:
find=pd.read_csv('finding_annotations.csv')
find.head()
columns = ['study_id', 'series_id', 'laterality', 
        'breast_birads', 'breast_density', 'finding_birads', 'xmin', 'ymin', 'xmax', 'ymax']
find = find[columns]

# birads to int
find['breast_birads'] = (
    find['breast_birads']
    .str.replace('BI-RADS ', '', regex=False)  # Remove prefix
    .astype(float)  # Convert to integer
)
find['finding_birads'] = (
    find['finding_birads']
    .str.replace('BI-RADS ', '', regex=False)
    .astype(float)
)

# casting to int
find['breast_density'] = find['breast_density'].map({'DENSITY A':1 , 'DENSITY B': 2 , 'DENSITY C' : 3 , 'DENSITY D': 4})


# 2. Calculate lesion size
find['width']  = find['xmax'] - find['xmin']
find['height'] = find['ymax'] - find['ymin']
find['area']   = find['width'] * find['height']


# Aggregate to breast-level
find = (
    find.groupby(['study_id','series_id', 'laterality'], as_index=False)
        .agg(
            breast_birads=('breast_birads', 'max'),          # highest BI-RADS across views
            breast_density=('breast_density', 'max'),        # highest density across views
            findings_count=('finding_birads', lambda x: x.notna().sum()),  # count only valid findings
            mean_width=('width', 'mean'),
            max_width=('width', 'max'),
            mean_height=('height', 'mean'),
            max_height=('height', 'max'),
            mean_area=('area', 'mean'),
            max_area=('area', 'max')
        )
)

print(find.shape)
find.head()


(978, 12)


,study_id,series_id,laterality,breast_birads,breast_density,findings_count,mean_width,max_width,mean_height,max_height,mean_area,max_area
0,003700f3c960e0b9bca2b8437c3dbf05,a729825009af3184f46ef773e3d93d87,L,4.0,3.0,2,59.369019,60.053009,58.139954,65.059937,3456.445210,3907.044982
1,008b8e61390fcb4c0873258c15b0a53a,bd0e137f3fc5b069db934797924d7d99,R,4.0,3.0,2,424.325073,443.100098,467.834472,476.609008,198678.638727,211185.498374
2,00a369b4ec1e5e0ff34e6bd838e5f2d6,0da08b11f65b222c697727fd32ab068e,L,3.0,3.0,1,288.879990,288.879990,325.830079,325.830079,94125.789833,94125.789833
3,012e0595adba5173b6e60a97f9e84b6e,a80a25f4bb029ba92a3ae1c6babcc767,L,3.0,3.0,2,428.645996,443.984009,523.434998,568.439942,225058.602514,252378.244211
4,012e0595adba5173b6e60a97f9e84b6e,a80a25f4bb029ba92a3ae1c6babcc767,R,3.0,3.0,2,374.260010,376.979980,422.390075,444.460083,158023.683900,165134.716572


In [4]:
meta = pd.read_csv('metadata.csv')
meta = meta.rename(columns={
    "Series Instance UID": "series_id",
    "Patient's Age": "age"
})

In [5]:
# Merge on study_id + laterality
df = pd.merge(breast, find, on=['study_id', 'laterality'], how='inner')

meta_agg = meta.groupby('series_id', as_index=False)['age'].max()
df = pd.merge(df, meta_agg, left_on='series_id_x', right_on='series_id', how='left')






# Check shape and sample
print("Merged shape:", df.shape)
# Map laterality: L -> 0, R -> 1
df['laterality'] = df['laterality'].map({'L': 0, 'R': 1})

df = df.drop(columns=['study_id', 'breast_birads_y','breast_density_y'])
df = df.rename(columns={
     'breast_birads_x': 'breast_birads',
    'breast_density_x': 'breast_density'
})
# Merge BI-RADS 2 into BI-RADS 3, then re-map
df['breast_birads'] = df['breast_birads'].replace({2: 3})
df['breast_birads'] = df['breast_birads'].replace({4: 5})
# as only 8 cases ofbirads 2
df['breast_birads'] = df['breast_birads'].map({3:0,5:1})

# ---------------------------------------------------------
# Remove trailing 'Y', leading zeros, and convert to numeric
df['age'] = pd.to_numeric(
    df['age'].astype(str).str.rstrip('Y').str.lstrip('0'),
    errors='coerce'
)

# Now drop rows with NaN ages
df = df.dropna(subset=['age'])

# Now it's safe: convert to int
df['age'] = df['age'].astype(int)


#-----------------------------------------------------------

# 4. Basic ratios & per-finding metrics
df['width_ratio']          = df['max_width']  / df['mean_width']
df['height_ratio']         = df['max_height'] / df['mean_height']
df['area_ratio']           = df['max_area']   / df['mean_area']
df['aspect_ratio']         = df['mean_width'] / df['mean_height']
df['lesion_density_ratio'] = df['findings_count'] / df['breast_density']
df['area_per_finding']     = df['mean_area'] / (df['findings_count'] + 1e-6)

# 5. Count & density interactions (scaled)
df['count_x_mean_area']    = (df['findings_count'] * df['mean_area'])  / 1000
df['count_x_max_area']     = (df['findings_count'] * df['max_area'])   / 1000
df['density_x_mean_area']  = (df['breast_density'] * df['mean_area'])  / 1000
df['density_x_max_area']   = (df['breast_density'] * df['max_area'])   / 1000

# 6. Age-based interactions & non-linear
df['age_x_density']        = df['age'] * df['lesion_density_ratio']
df['age_x_mean_area']      = (df['age'] * df['mean_area']) / 1000
df['age_x_aspect']         = df['age'] * df['aspect_ratio']
df['age_squared']          = df['age'] ** 2
df['log_age']              = np.log1p(df['age'])
df['age_group']            = pd.cut(df['age'], bins=[0, 40, 55, 100], labels=[0, 1, 2]).astype(int)

# 7. Area variability & shape complexity
df['area_variability']     = df['max_area']  - df['mean_area']
df['area_var_ratio']       = df['area_variability'] / (df['mean_area'] + 1)
df['shape_complexity']     = df['width_ratio'] * df['height_ratio'] * df['aspect_ratio']
df['elongation']           = np.maximum(df['width_ratio'], df['height_ratio'])
df['size_consistency']     = 1 / (df['width_ratio'] + df['height_ratio'] + df['area_ratio'])

# 8. Clinical risk features
df['birads_age_risk']      = (df['breast_birads'] * df['age']) / 50
df['density_age_interaction']= df['lesion_density_ratio'] * df['age']
df['area_per_age']         = df['mean_area'] / (df['age'] + 1)

# 9. Log transforms
for col in ['mean_area', 'max_area', 'count_x_mean_area', 'count_x_max_area']:
    df[f'log_{col}']      = np.log1p(df[col])

# 10. Select top features for modeling
top_features = [
    'breast_birads',
    'width_ratio',
    'age_x_aspect',
    'log_max_area',
    'area_variability',
    'aspect_ratio',
    'age_x_density',
    'height_ratio',
    'density_x_mean_area',
    'elongation',
    'size_consistency'
]

df = df[top_features]
df.head()

Merged shape: (1001, 17)


,breast_birads,width_ratio,age_x_aspect,log_max_area,area_variability,aspect_ratio,age_x_density,height_ratio,density_x_mean_area,elongation,size_consistency
0,1,1.011521,44.930149,8.270793,450.599772,1.021140,29.333333,1.119023,10.369336,1.119023,0.306663
1,1,1.044247,47.163912,12.260497,12506.859647,0.906998,34.666667,1.018756,596.035916,1.044247,0.319902
2,0,1.000000,43.443256,11.452398,0.000000,0.886597,16.333333,1.000000,282.377369,1.000000,0.333333
3,0,1.035782,39.307666,12.438688,27319.641697,0.818910,32.000000,1.085980,675.175808,1.085980,0.308342
4,0,1.007268,42.530546,12.014523,7111.032672,0.886053,32.000000,1.052250,474.071052,1.052250,0.322111


In [ ]:
# feature engineering

X = df.drop(columns=['breast_birads'])
y = df['breast_birads']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
# X and y from your preprocessed features
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale features for neural net
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, f1_score
# Compute class weights to balance classes
from sklearn.utils.class_weight import compute_class_weight





class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# Define improved model architecture
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(1, activation='sigmoid')
])

optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

# Set early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Train model with class weights
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=200,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=2
)

# Predict probabilities
test_probs = model.predict(X_test_scaled).flatten()

# Optimize threshold for best F1
precision, recall, thresholds = precision_recall_curve(y_test, test_probs)
f1_scores = 2 * precision * recall / (precision + recall)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f'Optimal threshold: {best_threshold:.3f} with F1: {f1_scores[best_idx]:.3f}')

# Predictions with optimized threshold
test_preds = (test_probs >= best_threshold).astype(int)

# Metrics
print(classification_report(y_test, test_preds, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(y_test, test_preds))
print('Test F1:', f1_score(y_test, test_preds))

c:\Users\harsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
22/22 - 8s - 378ms/step - accuracy: 0.5573 - loss: 0.7701 - val_accuracy: 0.6069 - val_loss: 0.6626
Epoch 2/200
22/22 - 0s - 16ms/step - accuracy: 0.6009 - loss: 0.7225 - val_accuracy: 0.6185 - val_loss: 0.6477
Epoch 3/200
22/22 - 0s - 16ms/step - accuracy: 0.6313 - loss: 0.6738 - val_accuracy: 0.6127 - val_loss: 0.6457
Epoch 4/200
22/22 - 0s - 16ms/step - accuracy: 0.5951 - loss: 0.7082 - val_accuracy: 0.6069 - val_loss: 0.6477
Epoch 5/200
22/22 - 0s - 16ms/step - accuracy: 0.6052 - loss: 0.6937 - val_accuracy: 0.6012 - val_loss: 0.6457
Epoch 6/200
22/22 - 0s - 16ms/step - accuracy: 0.6299 - loss: 0.6438 - val_accuracy: 0.6127 - val_loss: 0.6413
Epoch 7/200
22/22 - 0s - 17ms/step - accuracy: 0.6270 - loss: 0.6554 - val_accuracy: 0.6012 - val_loss: 0.6379
Epoch 8/200
22/22 - 0s - 14ms/step - accuracy: 0.6255 - loss: 0.6773 - val_accuracy: 0.6069 - val_loss: 0.6295
Epoch 9/200
22/22 - 0s - 10ms/step - accuracy: 0.6430 - loss: 0.6382 - val_accuracy: 0.6243 - val_loss: 0.6223
